# SQS суперячейки 128 атомов: Al-Co-Cr-Fe-Ni-Ti (L12)
Упрочняющая фаза **(Ni,Co,Fe,Cr)₃(Al,Ti)** со структурой **L12** (Pm-3m, #221)

Генерация **Special Quasirandom Structure (SQS)** с помощью **sqsgenerator**.
- Размер: **128 атомов** (4×4×2 суперячейка)
- Режим: `sublattice_mode: split` (строгое разделение A/B подрешёток)
- Итерации: **50 000 000** (высокая точность оптимизации SRO)

In [ ]:
%pip install sqsgenerator pymatgen

In [ ]:
from sqsgenerator import parse_config, optimize, write
from pymatgen.core import Structure

## 1. Параметры и генерация координат

In [ ]:
a = 3.57
nx, ny, nz = 4, 4, 2  # 4x4x2 -> 32 ячейки * 4 атома = 128 атомов

coords = []
species = []

for ix in range(nx):
    for iy in range(ny):
        for iz in range(nz):
            # B-сайт (1a)
            coords.append([ix/nx, iy/ny, iz/nz])
            species.append("Al")
            # A-сайты (3c)
            for offset in [[0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]]:
                coords.append([(ix+offset[0])/nx, (iy+offset[1])/ny, (iz+offset[2])/nz])
                species.append("Ni")

print(f"Сгенерировано сайтов: {len(coords)}")

## 2. Конфигурация sqsgenerator

In [ ]:
config = {
    "iterations": 50000000,
    "sublattice_mode": "split",
    "structure": {
        "lattice": [[a*nx, 0, 0], [0, a*ny, 0], [0, 0, a*nz]],
        "coords": coords,
        "species": species,
        "supercell": [1, 1, 1]
    },
    "composition": [
        # B-подрешётка (32 сайта): Al:Ti = 3:1 -> 24 Al, 8 Ti
        {"sites": "Al", "Al": 24, "Ti": 8},
        # A-подрешётка (96 сайтов): Ni:Co:Fe:Cr = 12:5:4:3 -> 48 Ni, 20 Co, 16 Fe, 12 Cr
        {"sites": "Ni", "Ni": 48, "Co": 20, "Fe": 16, "Cr": 12}
    ],
    "max_results_per_objective": 5
}

print("Конфигурация готова. Запуск оптимизации...")
print("(50M итераций на C++ займут ~1-3 минуты)")

## 3. Запуск оптимизации SQS

In [ ]:
cfg = parse_config(config)
pack = optimize(cfg)

print(f"\nНайдено решений: {len(pack)}")
print("Топ-5 конфигураций по Objective (ниже = лучше):")
for i, (obj, sols) in enumerate(pack[:5]):
    print(f"  #{i}: {obj:.6f} ({len(sols)} структур)")

## 4. Экспорт лучшего SQS

In [ ]:
best = pack.best()
print(f"Лучший objective: {best.objective:.6f}")

# Экспорт в CIF (формат определяется расширением)
write(best.structure(), "HEA_L12_SQS_128atoms.cif")
print("Сохранен: HEA_L12_SQS_128atoms.cif")

## 5. Проверка структуры

In [ ]:
sqs_128 = Structure.from_file("HEA_L12_SQS_128atoms.cif")

print(f"Формула: {sqs_128.composition.formula}")
print(f"Число атомов: {sqs_128.num_sites}")
print(f"Решётка: {sqs_128.lattice.a:.2f} x {sqs_128.lattice.b:.2f} x {sqs_128.lattice.c:.2f}")
print(f"\nСостав:")
for el, amt in sqs_128.composition.as_dict().items():
    print(f"  {el}: {amt:.0f} ({amt/sqs_128.num_sites*100:.1f}%)")

In [ ]:
print("Первые 10 позиций (дробные координаты):")
print(f"{'#':>3}  {'Element':>7}  {'x':>10}  {'y':>10}  {'z':>10}")
print("-" * 45)
for i, site in enumerate(sqs_128[:10]):
    fc = site.frac_coords
    print(f"{i+1:>3}  {site.species_string:>7}  {fc[0]:>10.4f}  {fc[1]:>10.4f}  {fc[2]:>10.4f}")
print("...")